![image.png](https://i.imgur.com/a3uAqnb.png)

# 🎥 UCF101 Video Classification Using 3D ResNet - Homework Assignment

In this homework, you will fine tune a **3D ResNet model for video classification** to recognize human actions from short video clips. This project is designed to give you practical experience working with spatiotemporal data using deep learning.

## 📌 Project Overview
- **Task**: Classify human actions from video clips
- **Architecture**: 3D ResNet for video classification
- **Dataset**: UCF101 action recognition dataset
- **Goal**: Build an accurate classification model using PyTorch

## 📚 Learning Objectives
By completing this assignment, you will:
- Learn how to process and sample frames from video files
- Fine tune a 3D ResNet model using PyTorch
- Train and evaluate models on the UCF101 dataset
- Analyze model performance on real-world video data

## 📦 Dataset Details
- The **UCF101** dataset contains 13,320 training videos across 101 action categories
- Each video clip captures a single action performed by a human in various environments
- Videos are processed into fixed-length clips (e.g., 16 or 32 frames) for training

## 🏷️ Labels Representation
- Each video is labeled with a class index from `0` to `100`
- Example classes include:
  - `Basketball`
  - `Horse Riding`
  - `Ice Dancing`
  - `Playing Guitar`
  - `Volleyball Spiking`

## 🔗 Reference
Original dataset link: [UCF101 Kaggle Dataset](https://www.kaggle.com/datasets/pevogam/ucf101)


## 1️⃣ Initial Setup and Library Installation

**Task**: Set up the environment and install necessary libraries.

In [1]:
from IPython.display import clear_output

In [2]:
# Incase you run this notebook outside colab (where the libraries aren't already pre-installed)

# %pip install torch
# %pip install matplotlib
# %pip install scikit-learn
# %pip install pandas
# %pip install numpy
# %pip install opencv-python

clear_output()

## 2️⃣ Import Libraries and Configuration

**Task**: Import all necessary libraries and set up configuration parameters.

**Requirements**:
- Import PyTorch and 3D ResNet model
- Import data processing libraries (pandas, sklearn, cv2)
- Import visualization libraries
- Set random seeds for reproducibility
- Configure hyperparameters with reasonable values

In [3]:
import os
import cv2
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import torch.nn.functional as F
import sklearn as sk


import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torchvision.models.video import r3d_18
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

import kagglehub
from IPython.display import clear_output

In [4]:
# TODO: Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
# TODO: Set device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
BATCH_SIZE = 8               # يعتمد على حجم الذاكرة (في Colab أو 8-16)
LEARNING_RATE = 1e-4         # بداية جيدة للتعلم
NUM_EPOCHS = 15              # جيد كبداية لتجربة fine-tuning
FRAMES_PER_CLIP = 16         # مدعوم رسميًا من r3d_18
NUM_CLASSES = 101            # عدد التصنيفات في UCF101
VALIDATION_SPLIT = 0.1       # 10% من البيانات للتقييم
TEST_SPLIT = 0.1             # 10% للاختبار (إذا فيه مجموعة اختبار منفصلة)


Using device: cuda


## 3️⃣ Data Loading and Exploration

**Task**: Download the UCF101 dataset and explore its structure.

**Requirements**:
- Download and load the dataset
- Display basic information about the data
- Load class mappings and video file paths
- Understand the dataset structure and labels

In [5]:
path = kagglehub.dataset_download("pevogam/ucf101")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/ucf101


In [6]:
# Load class name to index
classInd_txt = path + "/UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist/classInd.txt"
cls_df = pd.read_csv(classInd_txt, sep=" ", header=None,
                        names=["index", "class_name"])
cls_df.head()

,index,class_name
0,1,ApplyEyeMakeup
1,2,ApplyLipstick
2,3,Archery
3,4,BabyCrawling
4,5,BalanceBeam


In [8]:
class_to_idx = {}
idx_to_class = {}

classInd_txt = path + "/UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist/classInd.txt"

with open(classInd_txt, "r") as f:
    for line in f:
        idx, class_name = line.strip().split()
        idx = int(idx) - 1  # convert to 0-based indexing
        class_to_idx[class_name] = idx
        idx_to_class[idx] = class_name

print(f"There are {len(class_to_idx)} classes.\n")
print(class_to_idx)
print()
print(idx_to_class)

There are 101 classes.

{'ApplyEyeMakeup': 0, 'ApplyLipstick': 1, 'Archery': 2, 'BabyCrawling': 3, 'BalanceBeam': 4, 'BandMarching': 5, 'BaseballPitch': 6, 'Basketball': 7, 'BasketballDunk': 8, 'BenchPress': 9, 'Biking': 10, 'Billiards': 11, 'BlowDryHair': 12, 'BlowingCandles': 13, 'BodyWeightSquats': 14, 'Bowling': 15, 'BoxingPunchingBag': 16, 'BoxingSpeedBag': 17, 'BreastStroke': 18, 'BrushingTeeth': 19, 'CleanAndJerk': 20, 'CliffDiving': 21, 'CricketBowling': 22, 'CricketShot': 23, 'CuttingInKitchen': 24, 'Diving': 25, 'Drumming': 26, 'Fencing': 27, 'FieldHockeyPenalty': 28, 'FloorGymnastics': 29, 'FrisbeeCatch': 30, 'FrontCrawl': 31, 'GolfSwing': 32, 'Haircut': 33, 'Hammering': 34, 'HammerThrow': 35, 'HandstandPushups': 36, 'HandstandWalking': 37, 'HeadMassage': 38, 'HighJump': 39, 'HorseRace': 40, 'HorseRiding': 41, 'HulaHoop': 42, 'IceDancing': 43, 'JavelinThrow': 44, 'JugglingBalls': 45, 'JumpingJack': 46, 'JumpRope': 47, 'Kayaking': 48, 'Knitting': 49, 'LongJump': 50, 'Lunges':

In [9]:
videos_root = path + "/UCF101/UCF-101"
paths_file = path + "/UCF101TrainTestSplits-RecognitionTask/ucfTrainTestlist/trainlist01.txt"

# Initialize lists for videos and labels
all_videos = []
all_labels = []

# Read training list file and extract video paths and labels
with open(paths_file, "r") as f:
    for line in f:
        video_path = line.strip().split()[0]  # Format: class_name/video_file.avi
        class_name = video_path.split("/")[0]  # Extract class name from path
        full_path = f"{videos_root}/{video_path}"
        label = class_to_idx[class_name]

        all_videos.append(full_path)
        all_labels.append(label)

# Print total videos loaded
print(f"Total videos loaded: {len(all_videos)}")


Total videos loaded: 9537


## 4️⃣ Data Preprocessing & Dataset Creation & data showing

**Task**: Create custom dataset class and prepare data splits.

**Requirements**:
- Implement video reading and frame sampling
- Create custom PyTorch Dataset class
- Split data into train/validation/test sets
- Apply appropriate transformations
- Create data loaders for batch processing
- Visualize clips of a video
- Show a video

In [10]:
def read_video_opencv(video_path, frames_per_clip=16):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < 1:
        cap.release()
        raise ValueError(f"Video {video_path} is empty or unreadable.")

    # Calculate indices to sample evenly
    indices = np.linspace(0, total_frames - 1, frames_per_clip).astype(int)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

    cap.release()

    # If video too short, pad with last frame
    while len(frames) < frames_per_clip:
        frames.append(frames[-1])

    # Convert to tensor: (T, H, W, C) -> (C, T, H, W)
    clip = np.stack(frames)
    clip = torch.tensor(clip).permute(3, 0, 1, 2).float() / 255.0  # Normalize [0,1]

    return clip

In [11]:
class UCF101Dataset(Dataset):
    def __init__(self, videos, labels, transform=None, frames_per_clip=16):
        self.videos = videos
        self.labels = labels
        self.transform = transform
        self.frames_per_clip = frames_per_clip

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        video_path = self.videos[idx]
        label = self.labels[idx]

        clip = read_video_opencv(video_path, self.frames_per_clip)

        if self.transform:
            clip = self.transform(clip)

        return clip, label

In [12]:
from sklearn.model_selection import train_test_split

# Split into train (70%) and temp (30%)
train_videos, temp_videos, train_labels, temp_labels = train_test_split(
    all_videos, all_labels, test_size=0.3, stratify=all_labels, random_state=42)

# Split temp into val (15%) and test (15%)
val_videos, test_videos, val_labels, test_labels = train_test_split(
    temp_videos, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42)

# Print dataset sizes
print("Train size:", len(train_videos))
print("Validation size:", len(val_videos))
print("Test size:", len(test_videos))


Train size: 6675
Validation size: 1431
Test size: 1431


In [13]:
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Define image transformations (resize only)
transform = lambda clip: F.interpolate(clip, size=(128, 128), mode='bilinear')

# Create dataset instances
train_dataset = UCF101Dataset(train_videos, train_labels, transform=transform, frames_per_clip=16)
val_dataset = UCF101Dataset(val_videos, val_labels, transform=transform, frames_per_clip=16)
test_dataset = UCF101Dataset(test_videos, test_labels, transform=transform, frames_per_clip=16)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Test data loader
videos, labels = next(iter(train_loader))
print("Batch shape:", videos.shape)
print("Labels:", labels)

# Get one sample
sample_video, sample_label = train_dataset[0]
print("Sample video shape:", sample_video.shape)
print(f"Sample label: {sample_label} ({idx_to_class[sample_label]})")


Batch shape: torch.Size([8, 3, 16, 128, 128])
Labels: tensor([39, 46,  0, 13, 58, 14, 20, 62])
Sample video shape: torch.Size([3, 16, 128, 128])
Sample label: 71 (PushUps)


In [14]:
def show_video_clip(tensor_clip, label='', fps=None):
    """
    Display a (C, T, H, W) tensor as an animation inline with a label.

    Parameters:
      tensor_clip (Tensor): A tensor of shape (C, T, H, W)
      label (str): Text label to display (e.g., class name)
      fps (int): Frames per second. If None, defaults to T (one second playback)
    """
    # Permute from (C, T, H, W) to (T, H, W, C)
    clip = tensor_clip.permute(1, 2, 3, 0).cpu().numpy()

    if fps is None:
        fps = clip.shape[0]  # Use T as default fps

    fig, ax = plt.subplots()
    ax.set_title(label)
    im = ax.imshow(clip[0])

    def update(frame_idx):
        im.set_array(clip[frame_idx])
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=clip.shape[0], interval=1000 // fps, blit=True)
    plt.close(fig)

    video_html = ani.to_html5_video()
    video_html = video_html.replace("controls>", "controls autoplay muted>")
    return HTML(video_html)



In [15]:
# TODO: Randomly get one video and show all its clips
rand_idx = random.randint(0, len(train_dataset) - 1)
video_clip, label = train_dataset[rand_idx]

In [16]:
# TODO: show the video using show_video_clip

print(f"Label: {label} ({idx_to_class[label]})")
show_video_clip(video_clip, label=idx_to_class[label])


Label: 22 (CricketBowling)


## 5️⃣ Model Architecture and Fine-Tuning Setup

**Task**: Set up 3D ResNet model for fine-tuning and define training functions.

**Requirements**:
- Load pre-trained 3D ResNet model
- Modify final layer for UCF101 classes
- Define training and evaluation functions
- Implement metrics calculation

In [17]:
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def build_model(num_classes):
    weights = R3D_18_Weights.DEFAULT
    model = r3d_18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def compute_metrics(preds, labels, num_classes):
    preds = preds.cpu().argmax(dim=1)
    labels = labels.cpu()

    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average='macro', zero_division=0)
    recall = recall_score(labels, preds, average='macro', zero_division=0)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)

    return {
        'accuracy': round(accuracy * 100, 2),
        'precision': round(precision * 100, 2),
        'recall': round(recall * 100, 2),
        'f1_score': round(f1 * 100, 2)
    }

# Build model and move to device
model = build_model(NUM_CLASSES).to(device)

# Optimizer & Loss
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# Model Info
print("Model loaded on", device)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")


Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth
100%|██████████| 127M/127M [00:00<00:00, 210MB/s]


Model loaded on cuda
Total parameters: 33,218,085


## 6️⃣ Training and Validation Functions

**Task**: Implement comprehensive training and validation loops.

**Requirements**:
- Define training loop with gradient computation
- Implement validation loop with metrics tracking
- Monitor training progress with appropriate metrics

In [19]:
def train_one_epoch(model, train_loader, optimizer, criterion, device, num_classes):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(train_loader, desc="Training", leave=False)

    for videos, labels in pbar:
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * videos.size(0)
        all_preds.append(outputs.detach())
        all_labels.append(labels)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    metrics = compute_metrics(all_preds, all_labels, num_classes)
    avg_loss = total_loss / len(train_loader.dataset)
    metrics['loss'] = round(avg_loss, 4)

    return metrics


In [21]:
@torch.no_grad()
def evaluate(model, val_loader, criterion, device, num_classes):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(val_loader, desc="Validating", leave=False)

    for videos, labels in pbar:
        videos = videos.to(device)
        labels = labels.to(device)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * videos.size(0)
        all_preds.append(outputs)
        all_labels.append(labels)

    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    metrics = compute_metrics(all_preds, all_labels, num_classes)
    avg_loss = total_loss / len(val_loader.dataset)
    metrics['loss'] = round(avg_loss, 4)

    return metrics




## 7️⃣ Training Loop with Validation

**Task**: Execute the complete training process with validation monitoring.

**Requirements**:
- Train model for specified epochs
- Track training and validation metrics
- Save best model based on validation performance
- Display training progress

In [ ]:
import copy

# Initialize tracking lists and best validation accuracy
train_losses = []
val_losses = []
best_val_acc = 0
best_model_state = None

for epoch in range(NUM_EPOCHS):
    print(f"\n Epoch {epoch+1}/{NUM_EPOCHS}")

    # Train for one epoch
    train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device, NUM_CLASSES)
    train_losses.append(train_metrics['loss'])
    print(f" Train - Loss: {train_metrics['loss']:.4f}, Acc: {train_metrics['accuracy']}%, F1: {train_metrics['f1_score']}%")

    # Evaluate on validation set
    val_metrics = evaluate(model, val_loader, criterion, device, NUM_CLASSES)
    val_losses.append(val_metrics['loss'])
    print(f"Val   - Loss: {val_metrics['loss']:.4f}, Acc: {val_metrics['accuracy']}%, F1: {val_metrics['f1_score']}%")

    #  Save best model based on validation accuracy
    if val_metrics['accuracy'] > best_val_acc:
        best_val_acc = val_metrics['accuracy']
        best_model_state = copy.deepcopy(model.state_dict())
        print(" Best model updated.")

#  Load best model after training
if best_model_state:
    model.load_state_dict(best_model_state)
    print(f"\n Best model restored with validation accuracy: {best_val_acc}%")



 Epoch 1/15


Training:   0%|          | 0/835 [00:00<?, ?it/s]

 Train - Loss: 1.7867, Acc: 70.97%, F1: 70.28%


Validating:   0%|          | 0/179 [00:00<?, ?it/s]

Val   - Loss: 0.3165, Acc: 93.78%, F1: 93.15%
 Best model updated.

 Epoch 2/15


...

Training:   0%|          | 0/835 [00:00<?, ?it/s]

 Train - Loss: 0.0693, Acc: 99.24%, F1: 99.22%


Validating:   0%|          | 0/179 [00:00<?, ?it/s]

Val   - Loss: 0.2366, Acc: 94.13%, F1: 94.18%

 Epoch 5/15


Training:   0%|          | 0/835 [00:00<?, ?it/s]

## 8️⃣ Model Evaluation and Testing

**Task**: Evaluate the trained model on the test set and calculate classification metrics.

**Requirements**:
- Load best model for testing
- Make predictions on test set
- Calculate comprehensive classification metrics
- Analyze model performance

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Get all predictions and labels
@torch.no_grad()
def get_all_preds_labels(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    for videos, labels in loader:
        videos = videos.to(device)
        outputs = model(videos)
        all_preds.append(outputs.argmax(dim=1).cpu())
        all_labels.append(labels.cpu())
    return torch.cat(all_preds), torch.cat(all_labels)

preds, labels = get_all_preds_labels(model, test_loader)

# Classification report
print("\nClassification Report:\n")
print(classification_report(labels, preds, target_names=list(class_to_idx.keys())))

# Confusion Matrix
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap="Blues", xticklabels=False, yticklabels=False)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## 9️⃣ Visualization and Analysis

**Task**: Create visualizations to analyze model performance and training progress.

**Requirements**:
- Plot training and validation loss curves
- Display sample predictions with confidence scores
- Analyze model performance across different classes

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F


plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='o')
plt.title("Training and Validation Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


model.eval()
sample_results = []

with torch.no_grad():
    for i in range(3):
        video_tensor, true_label = test_dataset[i]
        input_batch = video_tensor.unsqueeze(0).to(device)  # Add batch dimension

        output = model(input_batch)
        probs = F.softmax(output, dim=1)
        confidence, pred_idx = torch.max(probs, dim=1)
        pred_label = pred_idx.item()
        confidence = confidence.item()

        sample_results.append({
            'true_label': true_label,
            'true_class': idx_to_class[true_label],
            'predicted_class': idx_to_class[pred_label],
            'confidence': round(confidence * 100, 2),
            'video_tensor': video_tensor
        })


for i, result in enumerate(sample_results):
    print(f"\n Sample {i+1}")
    print(f" True Label     : {result['true_label']} ({result['true_class']})")
    print(f" Predicted Label: {result['predicted_class']} ({result['confidence']}% confidence)")

    display(show_video_clip(result['video_tensor'], label=f"Predicted: {result['predicted_class']} ({result['confidence']}%)"))


## 📝 Evaluation Criteria

Your homework will be evaluated based on:

1. **Implementation Correctness (50%)**
   - Proper 3D ResNet model setup and fine-tuning
   - Correct video data preprocessing and dataset implementation
   - Working training loop with validation
   - Appropriate loss function and optimizer usage

2. **Model Performance (25%)**
   - Reasonable classification metrics (Accuracy, Precision, Recall, F1)
   - Convergence during training
   - Generalization to test set

3. **Code Quality (25%)**
   - Clean, readable code with proper structure
   - Efficient video processing implementation
   - Good coding practices and documentation

# Created by: Hassan Alsayhah